In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source": "https://"}    
)

In [4]:
sample_doc

Document(metadata={'source': 'https://'}, page_content='Hello World')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

### Load .txt Data and Convert into LangChain Document

In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("Data/python.txt", encoding="utf-8")
document = loader.load()
document

C:\Users\sumit_mali\AppData\Local\Temp\ipykernel_26040\2136834160.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'Data/python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

### Load .pdf Data and Convert into LangChain Document using PyPDFLoader

In [7]:
from langchain_community.document_loaders import PyPDFLoader
pdf_loader = PyPDFLoader("Data/research2.pdf")
document = pdf_loader.load()

In [8]:
# from langchain_community.document_loaders import PyMuPDFLoader
# pdf_loader = PyMuPDFLoader("Data/research.pdf")
# document = pdf_loader.load()
# document

# A :-) Ingection PipeLine

#### 1.Data Convert Into Langchain Document Data------>Document

In [9]:
import os
from langchain_community.document_loaders.pdf import PyMuPDFLoader

In [10]:
def load_all_pdfs():  # Load all PDFs
    folder_path = "Data/pdfs"  # PDF folder path
    num_docs=0  # PDF count
    all_docs=[]  # Store all pages

    for filename in os.listdir(folder_path):  # Loop files
        if filename.lower().endswith(".pdf"):  # Check PDF
            #complete file Path
            pdf_path= os.path.join(folder_path,filename)  # Create full path
            loaders= PyMuPDFLoader(pdf_path)  # Create loader
            doc = loaders.load()  # Load PDF

            all_docs.extend(doc)  # Add pages
            num_docs += 1  # Increase count

    print("Total Numbers of Documents",num_docs)  # Show PDF count
    print("Total Pages",len(all_docs))  # Show page count
    return all_docs  # Return pages

In [11]:
all_pdf_document = load_all_pdfs()

Total Numbers of Documents 2
Total Pages 32


#### 2.Document Convert into Chunks    Document ----> Chunks

In [12]:
#!pip install langchain_text_splitters

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
def split_docs(document, chunk_size=500, chunk_overlap=50):  # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(  # Create text splitter
        chunk_size=chunk_size,  # Set chunk size
        chunk_overlap=chunk_overlap  # Set overlap size
    )
    chunks_docs = text_splitter.split_documents(document)  # Split documents
    return chunks_docs  # Return chunks

In [15]:
chunks = split_docs(all_pdf_document)
len(chunks)

76

####  3.Define Embedding Funtion

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
class EmbeddingManager:  # Manage embeddings
    def __init__(self,model_name="sentence-transformers/all-MiniLM-L6-v2"):  # Constructor
        self.model_name = model_name  # Store model name
        print("loading_model........",self.model_name)  # Show model name
        self.model= SentenceTransformer(self.model_name)  # Load model
        print("Embedding Dimentions=",self.model.get_sentence_embedding_dimension())  # Show embedding size

    def genrate_embeddings(self,text):  # Generate embeddings
        embedding = self.model.encode(text,show_progress_bar= True)  # Convert text to embeddings
        print("embedding shape:", embedding.shape)  # Show embedding shape
        return embedding  # Return embeddings

In [18]:
embedding_manager = EmbeddingManager()

loading_model........ sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Dimentions= 384


C:\Users\sumit_mali\AppData\Local\Temp\ipykernel_26040\954394949.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Dimentions=",self.model.get_sentence_embedding_dimension())  # Show embedding size


#### 4. Define The Vector DB Structure

In [19]:
import chromadb
import uuid

In [20]:
class VectorStoreManager:  # Manage vector store
    def __init__(self,persist_directory= "Data/vector_store",collection_name="pdf_documents"):  # Constructor
        self.collection_name= collection_name  # Store collection name
        self.persist_directory=persist_directory  # Store folder path
        self.collection = None  # Initialize collection
        self.client = None  # Initialize client

        self._initialize_store()  # Initialize store

    def _initialize_store(self):  # Setup vector store
        os.makedirs(self.persist_directory,exist_ok=True)  # Create folder
        # create a client

        self.client = chromadb.PersistentClient(path=self.persist_directory)  # Create ChromaDB client

        #create collection 
        self.collection = self.client.get_or_create_collection(  # Create collection
            name = self.collection_name,  # Collection name
            metadata={"description": "vectore store the collection for pdf embedding in RAG"}  # Collection info
        )

        print("initialize the vectore store With collection", self.collection_name)  # Show collection name
        print("Doc In collection:",self.collection.count())  # Show document count

    def add_documents(self, documents, embeddings):  # Add documents
        if len(documents) != len(embeddings):  # Check sizes
            raise ValueError("number of documents does not match number of embeddings")  # Stop if mismatch
    
        ids = []  # Store ids
        all_metadata = []  # Store metadata
        document_content = []  # Store text
        embeddings_list = []  # Store embeddings
    
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):  # Loop data
            doc_id = f"doc_{uuid.uuid4()}"  # Create unique id
            ids.append(doc_id)  # Save id
    
            metadata = dict(doc.metadata)  # Copy metadata
            metadata["doc_index"] = i  # Save index
            metadata["content_length"] = len(doc.page_content)  # Save text length
            all_metadata.append(metadata)  # Add metadata
    
            document_content.append(doc.page_content)  # Save text
            embeddings_list.append(embedding.tolist())  # Convert embedding
    
        self.collection.add(  # Add to collection
            ids=ids,
            metadatas=all_metadata,
            documents=document_content,
            embeddings=embeddings_list
        )
    
        print("total Document add in vector store =", len(document_content))  # Show added count
        print("docs in collection", self.collection.count())  # Show total count

In [21]:
vector_store = VectorStoreManager()

initialize the vectore store With collection pdf_documents
Doc In collection: 380


###### data=>Document=>chunks=>embedding=>store in vectore DB

####  5.Chunks document convet Into Embedding      Chunks -----> Embedding(Vector)

In [22]:
text=[doc.page_content for doc in chunks]  # Extract text from chunks

Embedded_chunks = embedding_manager.genrate_embeddings(text)  # Generate embeddings

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

embedding shape: (76, 384)


#### 6.Store this Embedding IN VectorsDB (CromaDB)

In [23]:
vector_store.add_documents(chunks,Embedded_chunks)

total Document add in vector store = 76
docs in collection 456


In [24]:
vector_store.add_documents(chunks,Embedded_chunks)  # Add chunks and embeddings to vector store

total Document add in vector store = 76
docs in collection 532


#### 7.Now search Query In Vector DB

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

In [26]:
class RAGRetriver:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_K=5, score_threshold=0.0):
        # query ===> embedding
        query_embedding = self.embedding_manager.genrate_embeddings([query])[0]
        
        # semantic Search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_K
        )

        # cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
            
            # MOVED OUTSIDE THE FOR LOOP
            print(f"retrieved {len(retrieved_docs)} Documents")
        else:
            print("No document Find")
        
        return retrieved_docs

In [27]:
rag_ritriver = RAGRetriver(embedding_manager,vector_store)

In [28]:
rag_ritriver.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 5 Documents


[{'id': 'doc_17d54303-249d-4cb1-8407-565206e5e6b8',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'format': 'PDF 1.3',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'modDate': "D:20180212212210-08'00'",
   'producer': 'PyPDF2',
   'trapped': '',
   'content_length': 112,
   'source': 'Data/pdfs\\research.pdf',
   'total_pages': 11,
   'keywords': '',
   'doc_index': 49,
   'creator': '',
   'file_path': 'Data/pdfs\\research.pdf',
   'creationdate': '',
   'creationDate': '',
   'title': 'Attention is All you Need',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'moddate': '2018-02-12T21:22:10-08:00',
   'page': 6},
  'distance': 0.7821961641311646,
  'similarity_score': 0.21780383586883545,
  'rank': 1},
 {'id': 'doc_0a8622b5-8057-4240-9c9e-9f7c740221f7',
  'document'

In [29]:
Query = input("Enter the Query: ") 
Final_result = rag_ritriver.retrieve(Query)
print("Result Is : \n", Final_result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 0 Documents
Result Is : 
 []


# TO Join LLM API

## 1. Chat_GPT

In [30]:
# !pip install langchain-openai

In [32]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    base_url="https://openrouter.ai/api/v1",
    model = "openai/gpt-4o-mini",
    temperature =0.1,
    max_tokens= 1024
)

In [33]:
# Genrate our retrival-augmented Output
def genrate_output(query , retriver,llm,top_k=3):
    results = retriver.retrieve(query,top_k)

    context = "n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We are not fount relevent context for given Query")

    # context + Query

    prompt = f""" use given context to genrate the ans for the query Qwery : {query} Context : {context}"""

    responce = llm.invoke(prompt) # expecting String as Prompt
    return responce.content

In [34]:
answer = genrate_output("What is RAG",rag_ritriver,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 0 Documents
We are not fount relevent context for given Query


In [35]:
print(answer)

RAG, or Retrieval-Augmented Generation, is a machine learning approach that combines the strengths of information retrieval and natural language generation. It involves retrieving relevant information from a large dataset or knowledge base and then using that information to generate coherent and contextually appropriate responses. This method enhances the ability of language models to provide accurate and informative answers by grounding their responses in real-world data. RAG is particularly useful in applications like chatbots, question-answering systems, and other scenarios where up-to-date and contextually relevant information is crucial.


## 2. Groq

In [36]:
# !pip install langchain-groq

In [38]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
     model="qwen/qwen3-32b",
    temperature =0.1,
    max_tokens= 1024
)

In [39]:
# Genrate our retrival-augmented Output
def genrate_output(query , retriver,llm,top_k=3):
    results = retriver.retrieve(query,top_k)

    context = "n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We are not fount relevent context for given Query")

    # context + Query

    prompt = f""" use given context to genrate the ans for the query Qwery : {query} Context : {context}"""

    responce = llm.invoke([prompt.format(context=context,query=query)]) # expecting list as a prompt
    return responce.content

In [1]:
answer = genrate_output("What is Rag",rag_ritriver,llm)

NameError: name 'genrate_output' is not defined

In [41]:
print(answer)

<think>
Okay, the user is asking about the Encoder Decoder in the context provided. Let me look at the context again. It mentions positional encodings in both the encoder and decoder stacks, and a dropout rate of 0.1 for the base model. The context is a bit repetitive, with the same lines repeated multiple times. 

First, I need to recall what an encoder-decoder architecture is. Typically, in models like Transformers, the encoder processes the input sequence, and the decoder generates the output sequence. Positional encodings are crucial here because they add information about the position of each token in the sequence, which the model otherwise wouldn't have since it's based on self-attention.

The context specifically says that positional encodings are used in both encoder and decoder stacks. The base model uses a dropout rate (Pdrop) of 0.1. Dropout is a regularization technique where randomly selected neurons are ignored during training, which helps prevent overfitting. So, applyin